# Task 1 - 0.501

## Imports

In [1]:
import numpy as np
import pandas as pd
import re
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.feature_extraction.text import TfidfVectorizer

## Dataset

In [2]:
train_df = pd.read_csv("train.csv", sep="\t", engine="python", quotechar='"')
test_df = pd.read_csv("test.csv", sep="\t", engine="python", quotechar='"')
print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")

Train shape: (139156, 3)
Test shape: (34790, 2)


## Preprocessing and Feature Building

In [3]:
# =========================================================
# 1. TEXT PREPROCESSING
# =========================================================
def preprocess_text(text):
    if not isinstance(text, str): # Checks to ensure text is a string, handles NaN and non-string inputs
        return ""
    text = text.lower() # Convert to lowercase
    text = re.sub(r"[^\w\s]", " ", text) # Remove punctuation (keep only words and whitespace)
    text = re.sub(r"\b\d+\b", " NUM ", text) # Replace standalone numbers with a placeholder
    text = re.sub(r"\s+", " ", text) # Replace multiple whitespace with a single space
    return text.strip() # Remove leading/trailing whitespace


# =========================================================
# 2. FEATURE BUILDING
# =========================================================
def build_features( # Build TF-IDF features
    X_train_text, # Training text data
    X_other_text, # Val/test text data
    ngram_range=(1, 2), # Use unigrams(1-word) and bigrams (2-word phrases)
    max_features=40000, # Limit to top 40k features by frequency
    min_df=2, # Reduce noise by ignoring terms that appear in fewer than 2 documents
    max_df=0.95, # Ignore very common terms that appear in more than 95% of documents
    use_stopwords=True # Remove common English stopwords to focus on more meaningful terms
):
    stop_words = "english" if use_stopwords else None # Use built-in English stopwords if specified

    vec = TfidfVectorizer( # Initialize TF-IDF vectorizer with specified parameters
        ngram_range=ngram_range, # Use specified n-gram range
        max_features=max_features, # Limit to top features
        min_df=min_df, # Minimum document frequency
        max_df=max_df, # Maximum document frequency
        sublinear_tf=True, # Uses Frequency scaling (1 + log(tf)) to reduce impact of very frequent terms
        analyzer="word", # Analyze at the word level
        stop_words=stop_words, # Remove stopwords if specified
        strip_accents="unicode", # Normalize accents to improve consistency
        token_pattern=r"\w{2,}" # Only keeps words with 2+ char
    )

    X_train = vec.fit_transform(X_train_text).astype(np.float32) # Fit learns vocab and IDF scores
    X_other = vec.transform(X_other_text).astype(np.float32) # Uses just transform to apply the learnt vocab and IDF to val/test data

    return X_train, X_other, vec


## Binary Logistic Regression from Scratch

In [4]:
class BinaryLogisticRegressionScratch:
    def __init__(self, reg_lambda=1e-4, class_weight="balanced", lr_decay=0.01, min_lr=1e-3, early_stopping=True, patience=10, tol=1e-4, verbose=False, random_state=42 ):
        self.reg_lambda = reg_lambda # L2 regularization strength
        self.class_weight = class_weight # Handle class imbalance with balanced weighting
        self.lr_decay = lr_decay # Learning rate decay factor
        self.min_lr = min_lr # Minimum learning rate to prevent it from becoming too small
        self.early_stopping = early_stopping # Enable early stopping based on validation loss
        self.patience = patience # Number of epochs to wait for improvement before stopping
        self.tol = tol # Minimum improvement to qualify as an improvement for early stopping
        self.verbose = verbose # Print training progress if True
        self.random_state = random_state # Random seed for reproducibility

        self.w = None # Weights will be initialized during training
        self.b = 0.0 # Bias term initialized to zero
        self.rng_ = np.random.default_rng(random_state) # Random number generator for reproducibility

        self.best_w_ = None # To store best weights for early stopping
        self.best_b_ = None # To store best bias for early stopping
        self.best_loss_ = np.inf # To track best loss for early stopping
        self.history_ = [] # To store training history (loss, val_loss, etc.)

    def sigmoid(self, z): 
        z = np.clip(z, -30, 30) # Clip input to prevent overflow in exp
        return 1.0 / (1.0 + np.exp(-z)) # Sigmoid function to convert logits to probabilities

    def loss(self, y, y_hat, sample_weights=None):
        eps = 1e-12
        y_hat = np.clip(y_hat, eps, 1 - eps)
        ce = -(y * np.log(y_hat) + (1 - y) * np.log(1 - y_hat))
        if sample_weights is not None:
            data_loss = np.mean(sample_weights * ce)
        else:
            data_loss = np.mean(ce)
        reg_loss = 0.5 * self.reg_lambda * np.sum(self.w ** 2) if self.w is not None else 0.0
        return data_loss + reg_loss
 
    def gradients(self, X, y, y_hat, sample_weights=None):
        m = len(y)
        error = y_hat - y
        if sample_weights is not None:
            error = error * sample_weights
        dw = (X.T @ error) / m
        db = np.mean(error)
        if self.reg_lambda > 0 and self.w is not None:
            dw += self.reg_lambda * self.w
        return dw, db

    def _compute_pos_weight(self, y): # Compute positive class weight for handling class imbalance
        if self.class_weight is None: # No weighting, return 1.0
            return 1.0
        pos = np.sum(y == 1) # Count positive samples
        neg = np.sum(y == 0) # Count negative samples
        if pos == 0: # Avoid division by zero if no positive samples, return 1.0 (no weighting)
            return 1.0
        pos_weight = neg / max(pos, 1) # Compute weight as ratio of negative to positive samples
        pos_weight = min(pos_weight, 10.0)  # Clip to avoid extreme weights
        return float(pos_weight) 


    def train(self, X, y, bs=32, epochs=100, lr=0.01, X_val=None, y_val=None):
        y = np.asarray(y).astype(np.float64)
        n_samples, n_features = X.shape

        # Initialize weights and bias
        self.w = self.rng_.normal(0.0, 0.001, size=n_features)
        self.b = 0.0

        # Compute positive class weight for imbalance handling
        pos_weight = self._compute_pos_weight(y)

        best_metric = np.inf
        bad_epochs = 0
        if y_val is not None:
            y_val = np.asarray(y_val).astype(np.float64)
        for epoch in range(epochs):
            # Learning rate decay
            lr_t = max(self.min_lr, lr / (1.0 + self.lr_decay * epoch))
            
            # Shuffle data
            indices = self.rng_.permutation(n_samples)

            # Mini-batch gradient descent
            for start in range(0, n_samples, bs):
                end = min(start + bs, n_samples)
                batch_idx = indices[start:end]

                Xb = X[batch_idx]
                yb = y[batch_idx]

                # Forward pass
                logits = Xb @ self.w + self.b
                y_hat = self.sigmoid(np.asarray(logits).ravel())

                # Apply class weights in gradient
                sample_weights = np.where(yb == 1, pos_weight, 1.0)
                
                dw, db = self.gradients(Xb, yb, y_hat, sample_weights=sample_weights)
                grad_w = np.asarray(dw).ravel()
                grad_b = db

                # Update parameters
                self.w -= lr_t * grad_w
                self.b -= lr_t * grad_b

            # Compute training loss
            logits = X @ self.w + self.b
            y_hat_full = self.sigmoid(np.asarray(logits).ravel())
            sample_weights_full = np.where(y == 1, pos_weight, 1.0)
            train_loss = self.loss(y, y_hat_full, sample_weights_full)

            # Validation loss if provided
            if X_val is not None and y_val is not None:
                val_logits = X_val @ self.w + self.b
                y_hat_val = self.sigmoid(np.asarray(val_logits).ravel())
                sample_weights_val = np.where(y_val == 1, pos_weight, 1.0)
                val_loss = self.loss(y_val, y_hat_val, sample_weights_val)
                monitor = val_loss
            else:
                val_loss = None
                monitor = train_loss

            # Store history
            self.history_.append({
                "epoch": epoch + 1,
                "lr": lr_t,
                "train_loss": train_loss,
                "val_loss": val_loss
            })

            # Verbose output
            if self.verbose and ((epoch + 1) % 10 == 0 or epoch == 0):
                if val_loss is None:
                    print(f"Epoch {epoch+1:>3}/{epochs} | lr={lr_t:.5f} | train_loss={train_loss:.6f}")
                else:
                    print(f"Epoch {epoch+1:>3}/{epochs} | lr={lr_t:.5f} | "
                          f"train_loss={train_loss:.6f} | val_loss={val_loss:.6f}")

            # Early stopping
            if monitor + self.tol < best_metric:
                best_metric = monitor
                bad_epochs = 0
                self.best_loss_ = monitor
                self.best_w_ = self.w.copy()
                self.best_b_ = float(self.b)
            else:
                bad_epochs += 1

            if self.early_stopping and bad_epochs >= self.patience:
                if self.verbose:
                    print(f"Early stopping at epoch {epoch+1}")
                break

        # Restore best weights
        if self.best_w_ is not None:
            self.w = self.best_w_
            self.b = self.best_b_

        return self

    def predict_proba(self, X):
        logits = X @ self.w + self.b
        p1 = self.sigmoid(np.asarray(logits).ravel())
        p0 = 1.0 - p1
        return np.vstack([p0, p1]).T

    def predict(self, X, threshold=0.5):
        probs = self.predict_proba(X)[:, 1]
        return (probs >= threshold).astype(int)


## Sanity check for Binary Logistic Regression

In [5]:
from sklearn.datasets import make_classification

X_toy, y_toy = make_classification(n_samples=500, n_features=10, n_classes=2, random_state=42)

X_tr, X_val, y_tr, y_val = train_test_split(X_toy, y_toy, test_size=0.2, random_state=42)

model = BinaryLogisticRegressionScratch(random_state=42)
model.train(X_tr, y_tr, bs=32, epochs=50, lr=0.1, X_val=X_val, y_val=y_val)

y_pred = model.predict(X_val)

# Evaluate
print(f"\nAccuracy : {accuracy_score(y_val, y_pred):.4f}")
print(f"F1 Score : {f1_score(y_val, y_pred):.4f}")

# Check loss decreases over time
train_losses = [h["train_loss"] for h in model.history_]
print(f"\nFirst loss : {train_losses[0]:.6f}")
print(f"Last loss  : {train_losses[-1]:.6f}")
print(f"Loss decreased: {train_losses[-1] < train_losses[0]}")


Accuracy : 0.8800
F1 Score : 0.8723

First loss : 0.459453
Last loss  : 0.267294
Loss decreased: True


## One vs Rest Multi-class Adaptation

In [6]:
class OneVsRestScratch:

    def __init__(self, base_model_params=None):
        if base_model_params is None:
            base_model_params = {}
        self.base_model_params = base_model_params
        self.models_ = {}
        self.classes_ = None

    def fit(self, X, y, X_val=None, y_val=None):
        self.classes_ = np.sort(np.unique(y))
        self.models_ = {}

        print(f"Training OvR Logistic Regression for {len(self.classes_)} classes...")

        for i, cls in enumerate(self.classes_):
            print(f"Training class {cls} vs rest ({i+1}/{len(self.classes_)})")

            # Convert to binary problem: class vs rest
            y_binary = (y == cls).astype(int)

            if X_val is not None and y_val is not None:
                y_val_binary = (y_val == cls).astype(int)
            else:
                y_val_binary = None

            # Train binary classifier
            model = BinaryLogisticRegressionScratch(**self.base_model_params)
            model.train(X, y_binary, X_val=X_val, y_val=y_val_binary)

            self.models_[cls] = model

        return self

    def predict_proba(self, X): 
        all_probs = []

        for cls in self.classes_:
            # Get probability of positive class (current class)
            probs = self.models_[cls].predict_proba(X)[:, 1]
            all_probs.append(probs)

        return np.vstack(all_probs).T

    def predict(self, X): 
        prob_matrix = self.predict_proba(X)
        pred_idx = np.argmax(prob_matrix, axis=1)
        return self.classes_[pred_idx]

## Multi-class Adaptation on Train and Val Split

In [7]:
def run_validation(train_df): 
    df = train_df.copy()
    df["abstract"] = df["abstract"].fillna("").apply(preprocess_text)
    X_text = df["abstract"].to_numpy()
    y = df["label_id"].to_numpy()
    X_train_text, X_val_text, y_train, y_val = train_test_split(
        X_text,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )

    print(f"\n{'='*60}")
    print(f"PART 1b: Multi-class Logistic Regression (OvR)")
    print(f"Number of classes: {len(np.unique(y))}")
    print(f"{'='*60}")
    print("\nBuilding TF-IDF features for multiclass validation...")

    X_train, X_val, _ = build_features(
        X_train_text,
        X_val_text,
        ngram_range=(1, 2),
        max_features=40000,
        min_df=2,
        max_df=0.95,
        use_stopwords=True
    )

    # Create OvR model with binary classifiers
    model = OneVsRestScratch(
        base_model_params=dict(
            reg_lambda=1e-4,
            class_weight="balanced",
            lr_decay=0.01,
            min_lr=1e-3,
            early_stopping=True,
            patience=10,
            tol=1e-4,
            verbose=False,
            random_state=42
        )
    )

    # Train OvR model
    model.fit(X_train, y_train, X_val=X_val, y_val=y_val)
    
    # Predict on validation set
    y_pred = model.predict(X_val)

    # Evaluate
    acc = accuracy_score(y_val, y_pred)
    macro_f1 = f1_score(y_val, y_pred, average="macro")
    weighted_f1 = f1_score(y_val, y_pred, average="weighted")

    print("\n" + "="*60)
    print("Validation Results")
    print("="*60)
    print(f"Accuracy    : {acc:.4f}")
    print(f"F1 Macro    : {macro_f1:.4f}")
    print(f"F1 Weighted : {weighted_f1:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_val, y_pred, digits=4, zero_division=0))
    print("="*60)

    return model

run_validation(train_df)


PART 1b: Multi-class Logistic Regression (OvR)
Number of classes: 39

Building TF-IDF features for multiclass validation...
Training OvR Logistic Regression for 39 classes...
Training class 0 vs rest (1/39)
Training class 1 vs rest (2/39)
Training class 2 vs rest (3/39)
Training class 3 vs rest (4/39)
Training class 4 vs rest (5/39)
Training class 5 vs rest (6/39)
Training class 6 vs rest (7/39)
Training class 7 vs rest (8/39)
Training class 8 vs rest (9/39)
Training class 9 vs rest (10/39)
Training class 10 vs rest (11/39)
Training class 11 vs rest (12/39)
Training class 12 vs rest (13/39)
Training class 13 vs rest (14/39)
Training class 14 vs rest (15/39)
Training class 15 vs rest (16/39)
Training class 16 vs rest (17/39)
Training class 17 vs rest (18/39)
Training class 18 vs rest (19/39)
Training class 19 vs rest (20/39)
Training class 20 vs rest (21/39)
Training class 21 vs rest (22/39)
Training class 22 vs rest (23/39)
Training class 23 vs rest (24/39)
Training class 24 vs rest (

## Multi-class Adaptation on whole Dataset

In [8]:
def train_full_and_predict(train_df, test_df, output_path="LogReg_Prediction.csv"):
    train_df = train_df.copy()
    test_df = test_df.copy()

    print(f"\n{'='*60}")
    print(f"PART 1c: Full Training & Test Prediction")
    print(f"Output: {output_path}")
    print(f"{'='*60}")

    # Preprocess text
    print("\nPreprocessing abstracts...")
    train_df["abstract"] = train_df["abstract"].fillna("").apply(preprocess_text)
    test_df["abstract"] = test_df["abstract"].fillna("").apply(preprocess_text)

    X_train_text = train_df["abstract"].values
    y_train = train_df["label_id"].values
    X_test_text = test_df["abstract"].values

    print("\nBuilding TF-IDF features...")
    X_train, X_test, _ = build_features(
        X_train_text,
        X_test_text,
        ngram_range=(1, 2),
        max_features=40000,
        min_df=2,
        max_df=0.95,
        use_stopwords=True
    )

    print(f"Training data shape: {X_train.shape}")
    print(f"Test data shape: {X_test.shape}")
    print(f"Number of classes: {len(np.unique(y_train))}")
 
    model = OneVsRestScratch(
        base_model_params=dict(
            reg_lambda=1e-4,
            class_weight="balanced",
            lr_decay=0.01,
            min_lr=1e-3,
            early_stopping=True,
            patience=10,
            tol=1e-4,
            verbose=True,
            random_state=42
        )
    )

    print("\nTraining One-vs-Rest Logistic Regression on full training set...")
    model.fit(X_train, y_train)
    
    print("\nPredicting on test set...")
    test_pred = model.predict(X_test)

    # Create submission file
    submission = pd.DataFrame({
        "id": test_df["id"],
        "label_id": test_pred
    })

    submission.to_csv(output_path, index=False)

    print(f"\n✓ Saved final predictions to: {output_path}")
    print(f"\nSubmission file preview:")
    print(submission.head())
    print(f"\nPrediction distribution:")
    print(submission['label_id'].value_counts().sort_index())
    print("="*60)

    return submission

train_full_and_predict(train_df, test_df)


PART 1c: Full Training & Test Prediction
Output: LogReg_Prediction.csv

Preprocessing abstracts...

Building TF-IDF features...
Training data shape: (139156, 40000)
Test data shape: (34790, 40000)
Number of classes: 39

Training One-vs-Rest Logistic Regression on full training set...
Training OvR Logistic Regression for 39 classes...
Training class 0 vs rest (1/39)
Epoch   1/100 | lr=0.01000 | train_loss=0.873753
Epoch  10/100 | lr=0.00917 | train_loss=0.502339
Epoch  20/100 | lr=0.00840 | train_loss=0.418130
Epoch  30/100 | lr=0.00775 | train_loss=0.385812
Epoch  40/100 | lr=0.00719 | train_loss=0.368933
Epoch  50/100 | lr=0.00671 | train_loss=0.358603
Epoch  60/100 | lr=0.00629 | train_loss=0.351657
Epoch  70/100 | lr=0.00592 | train_loss=0.346631
Epoch  80/100 | lr=0.00559 | train_loss=0.342818
Epoch  90/100 | lr=0.00529 | train_loss=0.339852
Epoch 100/100 | lr=0.00503 | train_loss=0.337475
Training class 1 vs rest (2/39)
Epoch   1/100 | lr=0.01000 | train_loss=0.417071
Epoch  10/1

,id,label_id
0,173148,38
1,29098,4
2,28211,4
3,136101,0
4,97133,14
...,...,...
34785,36253,5
34786,122955,5
34787,74635,10
34788,73578,10
